# Sprint 3: explanations, price and sales estimates, and improvements over the baseline

**Parfois product similarity, MADSAD case study (FEP, University of Porto).**

Sprint 2 built the similarity engine (six blocks, a weighted mean of their similarities, weights learned from the team's
sorting). In Sprint 3 that engine is the **baseline**: it is frozen, and every change is measured against it. This
notebook covers:

| Section | What |
|---|---|
| 1 | The Sprint 2 engine, rebuilt (the baseline) |
| 2 | How a change is judged: the team's preferences, labels the change never saw, the guarded metrics |
| 3 | **Explanations** for every similar product: an exact decomposition of the score, and three checks |
| 4 | **Price and sales estimates** for a new product, with their explanations (SHAP) |
| 5 | Improvements tried against the baseline: fusion, attribute weights, image encoders (kept and rejected) |
| 6 | Photos of a person wearing the item *(in progress)* |

Like the other notebooks, it calls no project code: every step is written here with pandas, NumPy, matplotlib, Pillow and
LightGBM, reading only data files. Steps that need large neural networks (image and text embeddings) were run once; their
saved results are loaded. The notebook writes no file.

## 1. The Sprint 2 engine, rebuilt (the baseline)

The blocks are rebuilt exactly as in `notebooks/phase2_features.ipynb` (sections 1–10 there explain every step); the code
is repeated here so this notebook stands on its own. The colour palettes are loaded from `data/embeddings/palettes.parquet`
instead of being measured again on every photo (the Sprint 2 notebook shows they match).

In [1]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMG_DIR, EMB_DIR = ROOT / "data" / "images", ROOT / "data" / "embeddings"
KEY = "PROD_CLR_EQUIV"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

In [2]:
CODES = {c: str for c in ["PROD_REF", "PROD_REF_EQUIV", "SEA_COD", "CAT_COD", "GFA_COD", "GFS_COD"]}
items = pd.read_csv(ROOT / "outputs" / "phase1" / "items_clean.csv", encoding="utf-8-sig", dtype=CODES, low_memory=False)
items["STORE_DATE_FINAL"] = pd.to_datetime(items["STORE_DATE_FINAL"])
for c in ["img_trusted", "img_colour_trusted"]:
    items[c] = items[c].astype(str).eq("True")
items = items.sort_values(KEY, kind="stable").reset_index(drop=True)
print(f"{len(items):,} colourways x {items.shape[1]} columns | photos trusted: {items['img_trusted'].sum():,}")
items[[KEY, "CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN", "CLR_DES", "FINISHING", "COMPOSITION", "PRICE_BASE_W_VAT"]].head()

10,555 colourways x 116 columns | photos trusted: 9,445


,PROD_CLR_EQUIV,CAT_DES_EN,GFA_DES_EN,GFS_DES_EN,CLR_DES,FINISHING,COMPOSITION,PRICE_BASE_W_VAT
0,1328502PM,Jewellery,Rings,Cocktail,Pastel Multicolor,Worn Gold,Stone; Zinc,8.99
1,140428_BN,Jewellery,Necklaces,Pendant,Silver,Other,NaN,9.99
2,1404862PE,Jewellery,Key chains,Figures,Pearl,Rodhium,Acrylic; Crystal Glass; Iron; Zinc,12.99
3,140486_BM,Jewellery,Key chains,Figures,Bright Multicolor,Golden,Acrylic; Crystal Glass; Iron; Zinc,12.99
4,140486_HM,Jewellery,Key chains,Figures,Halftone Multicolor,NaN,Crystal Glass; Iron; Zinc,9.99


In [3]:
def normalise_rows(m):
    norms = np.linalg.norm(m, axis=1, keepdims=True)
    return np.divide(m, norms, out=np.zeros_like(m), where=norms > 0).astype(np.float32)

def shared_attributes(m, attrs, i, j):
    """How much each attribute adds to the similarity of rows i and j (the parts add up to the score)."""
    out = {}
    for name, (start, stop) in attrs.items():
        part = float(m[i, start:stop] @ m[j, start:stop])
        if part > 1e-6:
            out[name] = round(part, 3)
    return dict(sorted(out.items(), key=lambda kv: -kv[1]))

def neighbours(m, i, k=4, same_category=True):
    """Top k rows by cosine similarity: same category, other models only (another colour of the same model is
    trivially similar)."""
    s = m @ m[i]
    if same_category:
        s[(items["CAT_DES_EN"] != items["CAT_DES_EN"].iat[i]).to_numpy()] = -1
    s[(items["PROD_REF"] == items["PROD_REF"].iat[i]).to_numpy()] = -1
    top = np.argsort(-s, kind="stable")[:k]
    return top, s[top]

In [4]:
FINISH_FAMILY = {
    "gold": ["Golden", "Worn Gold", "Brushed Golden", "Antik Golden", "Gold Plated", "Matte Golden", "Burnish Golden",
             "Light Golden", "Worn Yellow Gold", "Matt Brushed Gold", "Scratched Gold"],
    "rose gold": ["Rose Gold", "Brushed Rose Gold", "Worn Rose God"],
    "silver": ["Rodhium", "Silver", "Worn Rhodium", "Worn Silver", "Brushed Silver", "Antik Silver", "Brushed Rhodium",
               "Matte Silver", "Silver Plated"],
    "black": ["Black Nikel", "Black Plating"],
    "other": ["Other", "Multifinishing", "Cooper"],
}
FAMILY_OF = {v: fam for fam, vals in FINISH_FAMILY.items() for v in vals}
items["FINISH_FAMILY"] = items["FINISHING"].map(FAMILY_OF)

# attribute -> (source columns forming a path, weight)
CATEGORICAL = {
    "CAT": (["CAT_DES_EN"], 1.0),
    "GFA": (["CAT_DES_EN", "GFA_DES_EN"], 3.0),
    "GFS": (["CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN"], 3.0),
    "CATEGORY_MATRIX": (["CATEGORY_MATRIX"], 1.5),
    "CLR_DES": (["CLR_DES"], 2.0),
    "CLR_TYPE": (["CLR_TYPE"], 0.5),
    "FINISHING": (["FINISHING"], 1.5),
    "FINISH_FAMILY": (["FINISH_FAMILY"], 1.0),
    "MATERIAL": (["MATERIAL"], 1.0),
    "PRINT_TYPE": (["PRINT_TYPE"], 1.0),
    "DIMENSION": (["DIMENSION"], 1.0),
    "OUTFIT": (["OUTFIT"], 1.0),
    "THEME": (["THEME"], 1.5),
    "FASHIONTYPE": (["FASHIONTYPE"], 0.5),
    "PROD_SEG": (["PROD_SEG"], 0.5),
    "NUMBER_OF_UNITS": (["NUMBER_OF_UNITS"], 0.5),
}

def categorical_block(df, spec=CATEGORICAL):
    parts, attrs, start = [], {}, 0
    for name, (cols, weight) in spec.items():
        values = df[cols[0]].astype("string")
        for c in cols[1:]:
            values = values + " › " + df[c].astype("string")       # missing if any part is missing
        levels = sorted(values.dropna().unique())
        codes = pd.Categorical(values, categories=levels).codes
        onehot = np.zeros((len(df), len(levels)), np.float32)
        onehot[np.flatnonzero(codes >= 0), codes[codes >= 0]] = np.sqrt(weight)
        parts.append(onehot)
        attrs[name] = (start, start + len(levels))
        start += len(levels)
    return normalise_rows(np.hstack(parts)), attrs

In [5]:
def composition_block(df):
    tokens = df["COMPOSITION"].fillna("").map(lambda s: sorted({t.strip() for t in str(s).split(";") if t.strip()}))
    vocab = sorted({t for ts in tokens for t in ts})
    col = {t: j for j, t in enumerate(vocab)}
    counts = pd.Series([t for ts in tokens for t in ts]).value_counts()
    idf = {t: float(np.log((1 + len(df)) / (1 + counts[t])) + 1) for t in vocab}
    m = np.zeros((len(df), len(vocab)), np.float32)
    for i, ts in enumerate(tokens):
        for t in ts:
            m[i, col[t]] = idf[t]
    return normalise_rows(m), {t: (j, j + 1) for t, j in col.items()}, counts, idf

comp, comp_attrs, counts, idf = composition_block(items)

In [6]:
PRICE_SIGMA, LAUNCH_SIGMA = 0.10, 1.5          # in percentile points / in months
NUMERIC = {"price": 2.0, "launch": 1.0, "season": 0.5}

def soft_bands(x, centres, sigma):
    m = np.exp(-((x[:, None] - centres[None, :]) ** 2) / (2 * sigma ** 2)).astype(np.float32)
    m[np.isnan(x)] = 0
    return normalise_rows(m)

def numeric_block(df):
    price_pct = df.groupby("CAT_DES_EN")["PRICE_BASE_W_VAT"].rank(pct=True, method="average").to_numpy(float)
    months = ((df["STORE_DATE_FINAL"] - df["STORE_DATE_FINAL"].min()).dt.days / 30.44).to_numpy(float)
    # centres run 3σ past both ends, so values at the edges are compared like the others
    price_c = np.arange(-3 * PRICE_SIGMA, 1 + 3 * PRICE_SIGMA + 1e-9, PRICE_SIGMA / 2)
    launch_c = np.arange(np.floor(np.nanmin(months) - 3 * LAUNCH_SIGMA), np.ceil(np.nanmax(months) + 3 * LAUNCH_SIGMA) + 1e-9,
                         LAUNCH_SIGMA / 2)
    seasons = sorted(df["SEA_COD"].dropna().unique())
    codes = pd.Categorical(df["SEA_COD"], categories=seasons).codes
    season = np.zeros((len(df), len(seasons)), np.float32)
    season[np.flatnonzero(codes >= 0), codes[codes >= 0]] = 1
    parts = {"price": soft_bands(price_pct, price_c, PRICE_SIGMA), "launch": soft_bands(months, launch_c, LAUNCH_SIGMA),
             "season": season}
    attrs, start = {}, 0
    for name, m in parts.items():
        attrs[name] = (start, start + m.shape[1])
        start += m.shape[1]
    return normalise_rows(np.hstack([m * np.float32(np.sqrt(NUMERIC[n])) for n, m in parts.items()])), attrs

In [7]:
def aligned(npy, meta_json):
    keys = json.loads((EMB_DIR / meta_json).read_text(encoding="utf-8"))["keys"]
    emb = np.load(EMB_DIR / npy).astype(np.float32)
    idx = items[KEY].map(pd.Series(range(len(keys)), index=keys))
    out = np.zeros((len(items), emb.shape[1]), np.float32)
    ok = idx.notna().to_numpy()
    out[ok] = emb[idx[ok].astype(int).to_numpy()]
    return out

image_raw = aligned("image_clip.npy", "image_clip.json")
palettes_saved = pd.read_parquet(EMB_DIR / "palettes.parquet").set_index(KEY)
palette_raw = np.stack([np.asarray(v, np.float32) if isinstance(v, (list, np.ndarray)) else np.zeros(24, np.float32)
                        for v in items[KEY].map(palettes_saved["palette_vector"])])
image_raw[~items["img_trusted"].to_numpy()] = 0
palette_raw[~items["img_trusted"].to_numpy()] = 0
yellow = items[KEY].map(pd.Series([sum(s * c[2] for s, c in zip(sh, lab)) for sh, lab in
                                   zip(palettes_saved["shares"], palettes_saved["lab"])], index=palettes_saved.index)).to_numpy()

VOTE_WEIGHTS = {"MATERIAL": {"categorical": 1.0, "composition": 3.0, "image": 1.0, "palette": 0.5},
                "FINISH_FAMILY": {"categorical": 1.0, "composition": 1.0, "image": 2.0, "palette": 2.0}}

def vote_features(target):
    spec = {k: v for k, v in CATEGORICAL.items() if k not in ("FINISHING", "MATERIAL", "FINISH_FAMILY")}
    parts = {"categorical": categorical_block(items, spec)[0], "composition": comp, "image": image_raw, "palette": palette_raw}
    w = VOTE_WEIGHTS[target]
    return normalise_rows(np.hstack([m * np.float32(np.sqrt(w[n])) for n, m in parts.items()]))

def knn_vote(Xq, Xt, yt, k=15):
    pred, conf = np.empty(len(Xq), dtype=object), np.zeros(len(Xq))
    for s in range(0, len(Xq), 512):
        sim = Xq[s:s + 512] @ Xt.T
        top = np.argsort(-sim, axis=1, kind="stable")[:, :k]
        for i, row in enumerate(top):
            votes = pd.Series(np.clip(sim[i, row], 0, None)).groupby(yt[row]).sum()
            if votes.sum() > 0:
                best = votes.sort_index().idxmax()
                pred[s + i], conf[s + i] = best, votes[best] / votes.sum()
    return pred, conf

def photo_agrees(pred, yel):
    pred, yel = np.asarray(pred, dtype=object), np.asarray(yel, float)
    return ~(((pred == "gold") & ~(yel >= 3.0)) | ((pred == "silver") & (yel > 6.0)))

In [8]:
TARGETS = {"MATERIAL": "Apparel", "FINISH_FAMILY": "Jewellery"}
filled, evidence = [], {}
for col, cat in TARGETS.items():
    X = vote_features(col)
    in_cat = items["CAT_DES_EN"].eq(cat)
    lab = (in_cat & items[col].notna()).to_numpy()
    y, groups = items.loc[lab, col].to_numpy(), items.loc[lab, "PROD_REF"].to_numpy()

    # cross-validation, folds by model
    rng = np.random.default_rng(0)
    models = np.unique(groups)
    fold = pd.Series(rng.integers(0, 5, len(models)), index=models).loc[groups].to_numpy()
    pred, conf = np.empty(len(y), dtype=object), np.zeros(len(y))
    for f in range(5):
        pred[fold == f], conf[fold == f] = knn_vote(X[lab][fold == f], X[lab][fold != f], y[fold != f])
    ok_photo = photo_agrees(pred, yellow[lab]) if col == "FINISH_FAMILY" else np.ones(len(y), bool)
    kept = (conf >= 0.8) & ok_photo
    prec = pd.Series(pred[kept] == y[kept]).groupby(pred[kept]).agg(["mean", "size"])
    allowed = set(prec.index[(prec["mean"] >= 0.9) & (prec["size"] >= 10)])
    evidence[col] = {"products with the value": len(y), "accuracy (all)": (pred == y).mean(),
                     "share kept (vote ≥ 80%)": kept.mean(), "accuracy (kept)": (pred == y)[kept].mean(),
                     "values allowed": ", ".join(sorted(allowed))}

    # fill: trusted photo, value missing
    todo = in_cat & items[col].isna() & items["img_trusted"]
    guess = pd.Series(np.nan, index=items.index[todo], dtype=object)
    method = pd.Series("", index=guess.index)
    if col == "MATERIAL":
        vocab = set(items.loc[in_cat, col].dropna())
        sib = items[items[col].notna()].groupby("PROD_REF")[col].agg(lambda s: s.iloc[0] if s.nunique() == 1 else np.nan)
        single = items["COMPOSITION"].fillna("").map(
            lambda s: [t.strip() for t in s.split(";") if t.strip()]).map(lambda t: t[0] if len(t) == 1 and t[0] in vocab else np.nan)
        for name, values in (("sibling", items["PROD_REF"].map(sib)), ("composition", single)):
            free = guess.isna() & values.loc[guess.index].notna()
            guess[free], method[free] = values.loc[guess.index[free]], name
    free = guess.index[guess.isna()]
    p, c = knn_vote(X[free.to_numpy()], X[lab], y)
    keep = (c >= 0.8) & np.isin(p, list(allowed))
    if col == "FINISH_FAMILY":
        keep &= photo_agrees(p, yellow[free.to_numpy()])
    guess[free[keep]], method[free[keep]] = p[keep], "neighbours"
    done = guess.notna()
    filled.append(pd.DataFrame({KEY: items.loc[guess.index[done], KEY], "column": col, "value": guess[done],
                                "method": method[done]}))
filled = pd.concat(filled, ignore_index=True)
print(pd.DataFrame(evidence).T.round(3).to_string())

              products with the value accuracy (all) share kept (vote ≥ 80%) accuracy (kept)                       values allowed
MATERIAL                         1303       0.785879                0.655411        0.923888  Cotton, Lyocell, Polyester, Viscose
FINISH_FAMILY                    4154       0.866875                0.616273        0.965234                  gold, other, silver


In [9]:
print(filled.groupby(["column", "method"]).size().rename("values filled").to_string())
for col in TARGETS:
    sub = filled[filled["column"] == col].set_index(KEY)["value"]
    items[col] = items[col].where(items[col].notna(), items[KEY].map(sub))
print("\nfilled now:", {col: f"{items.loc[items['CAT_DES_EN'] == cat, col].notna().mean():.0%} of {cat}"
                        for col, cat in TARGETS.items()})

column         method     
FINISH_FAMILY  neighbours     266
MATERIAL       composition    324
               neighbours     195
               sibling         61

filled now: {'MATERIAL': '83% of Apparel', 'FINISH_FAMILY': '95% of Jewellery'}


In [10]:
cat_m, cat_attrs = categorical_block(items)
num_m, num_attrs = numeric_block(items)
print(f"categorical {cat_m.shape} | composition {comp.shape} | numeric {num_m.shape}")

categorical (10555, 1624) | composition (10555, 95) | numeric (10555, 79)


In [11]:
text_m = normalise_rows(aligned("text.npy", "text.json"))
image_m = normalise_rows(image_raw)
palette_m = normalise_rows(palette_raw.copy())
q = items.index[items[KEY] == "167718_BU"][0]            # the example of the brief
blocks = {"categorical": cat_m, "composition": comp, "numeric": num_m, "text": text_m, "image": image_m}
print({b: m.shape for b, m in blocks.items()}, "| palettes", palette_m.shape)

{'categorical': (10555, 1624), 'composition': (10555, 95), 'numeric': (10555, 79), 'text': (10555, 384), 'image': (10555, 512)} | palettes (10555, 24)


In [12]:
palette_m[~items["img_colour_trusted"].to_numpy()] = 0
blocks["palette"] = palette_m
BLOCKS = ["image", "categorical", "text", "composition", "palette", "numeric"]
W_JUDGEMENT = {"image": 0.3, "categorical": 0.3, "text": 0.1, "composition": 0.1, "palette": 0.1, "numeric": 0.1}

has = {b: (np.abs(m).sum(1) > 0).astype(np.float32) for b, m in blocks.items()}
MODEL_CODES = [items[c].astype("string").fillna("").to_numpy(object) for c in ["PROD_REF", "PROD_REF_EQUIV"]]
CATEGORY_POOLS = [np.asarray(ix) for _, ix in sorted(items.groupby("CAT_DES_EN").indices.items())]

def score_matrix(w, qs, cs):
    """Scores of the queries qs (rows) against the candidates cs (columns): weights rescaled pair by pair."""
    num = np.zeros((len(qs), len(cs)), np.float32)
    den = np.zeros_like(num)
    for b, wb in w.items():
        if wb > 0:
            both = np.outer(has[b][qs], has[b][cs]) * np.float32(wb)
            num += both * (blocks[b][qs] @ blocks[b][cs].T)
            den += both
    return np.divide(num, den, out=np.zeros_like(num), where=den > 0)

def not_allowed(qs, cs):
    """The query itself and the other colours of its model."""
    out = qs[:, None] == cs[None, :]
    for v in MODEL_CODES:
        out |= (v[qs][:, None] == v[cs][None, :]) & (v[qs][:, None] != "")
    return out

def top_k(w, queries=None, k=10):
    """The k best candidates of every query (or only of `queries`, row numbers): one row per (query, rank)."""
    wanted = np.ones(len(items), bool) if queries is None else np.isin(np.arange(len(items)), queries)
    out = []
    for pool in CATEGORY_POOLS:
        qs_all = pool[wanted[pool]]
        for start in range(0, len(qs_all), 1024):              # 1,024 queries at a time keeps the memory small
            qs = qs_all[start:start + 1024]
            s = np.round(score_matrix(w, qs, pool).astype(np.float64), 6)
            s[not_allowed(qs, pool)] = -np.inf
            order = np.argsort(-s, axis=1, kind="stable")[:, :k]  # the pool is in key order: ties go to the lower key
            best = np.take_along_axis(s, order, axis=1)
            r, c = np.nonzero(np.isfinite(best))
            out.append(pd.DataFrame({"q": qs[r], "rank": c + 1, "c": pool[order[r, c]], "score": best[r, c]}))
    return pd.concat(out, ignore_index=True)

def contributions(w, i, j):
    """Each block's share of the score of the pair (i, j); the shares add up to the score."""
    used = [b for b in BLOCKS if w[b] > 0 and has[b][i] and has[b][j]]
    total = sum(w[b] for b in used)
    return {b: w[b] * float(blocks[b][i] @ blocks[b][j]) / total for b in used}

The baseline weights are the Sprint 2 result (25% blend between the tuned and the team-learned weights,
`src/phase3_weights.json`):

In [13]:
W_BASE = {"image": 0.4245, "categorical": 0.3317, "text": 0.0767, "composition": 0.0671, "palette": 0.034, "numeric": 0.066}
t0 = time.time()
base_nb = top_k(W_BASE, k=10)
print(f"top 10 of {base_nb['q'].nunique():,} products in {time.time() - t0:.0f} s")

top 10 of 10,555 products in 9 s


## 2. How a change is judged

There is no ground truth, so three signals are used, strongest first:

1. **The team's judgements.** In Sprint 2 each member sorted the candidates of their queries into *similar* (best first)
   and *not similar*: 138 sorted queries, about 4,000 pairwise preferences ("a is more similar to the query than b"). A
   version of the engine is scored by the share of preferences it orders right. Two ways: with the weights fixed, and with
   **5-fold cross-validation over queries** where the weights are refitted on 4 folds and scored on the 5th. Uncertainty:
   a **paired bootstrap over queries** (the preferences of one query are not independent).
2. **Labels a change never saw.** E.g. for an image encoder: do a photo's 4 nearest photos share its sub-family,
   material and finish? The encoder only sees pixels, so these labels are a fair test.
3. **The Sprint 2 signals on the test half of the models**, as guards: a change may not lower any of them by more than
   1 point. And **the brief's own example**: where the engine ranks the 4 products Parfois gave for `167718_BU`.

A ceiling to keep in mind: two team members agree with each other on about 70% of the pairs they both sorted.

In [14]:
refs = np.sort(items["PROD_REF"].astype(str).unique())
tune_refs = set(np.random.default_rng(0).permutation(refs)[: len(refs) // 2])
tune = items["PROD_REF"].astype(str).isin(tune_refs).to_numpy()
TUNE_Q, TEST_Q = np.flatnonzero(tune), np.flatnonzero(~tune)

# "known similar": same category › family › sub-family, theme and colour, shared by at least two models
GROUP_COLS = ["CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN", "THEME", "CLR_DES"]
full = items[GROUP_COLS].notna().all(axis=1)
gid = pd.Series(-1, index=items.index)
gid[full] = items.loc[full].groupby(GROUP_COLS, sort=True).ngroup()
models_in_group = items.loc[full].groupby(gid[full])["PROD_REF"].transform("nunique")
gid[full & models_in_group.reindex(items.index).lt(2)] = -1
gid = gid.to_numpy()

# the 20 nearest photos of every product with a trusted photo
photo = has["image"] > 0
visual_pool = {}
for pool in CATEGORY_POOLS:
    pool = pool[photo[pool]]
    for start in range(0, len(pool), 1024):
        qs = pool[start:start + 1024]
        s = np.round((image_m[qs] @ image_m[pool].T).astype(np.float64), 6)
        s[not_allowed(qs, pool)] = -np.inf
        for qi, row in zip(qs, np.argsort(-s, axis=1, kind="stable")[:, :20]):
            visual_pool[qi] = set(pool[row])

material_csv = items["MATERIAL"].where(~items[KEY].isin(filled.loc[filled["column"] == "MATERIAL", KEY]))
AGREE = {"same_family": items["GFA_DES_EN"],
         "same_subfamily": items["GFA_DES_EN"].astype("string") + " › " + items["GFS_DES_EN"].astype("string"),
         "same_colour": items["CLR_DES"], "same_finishing": items["FINISHING"], "same_material": material_csv}
AGREE = {n: v.astype("string").to_numpy(object, na_value=None) for n, v in AGREE.items()}
sales = items["sales_pct_in_cat"].to_numpy(float)
price = items["PRICE_BASE_W_VAT"].to_numpy(float)

def evaluate(nb):
    """Every signal for one set of neighbours (columns q, rank, c)."""
    top = nb[nb["rank"] <= 4]
    qi, ci = top["q"].to_numpy(), top["c"].to_numpy()
    m = {}
    known = gid[qi] >= 0
    m["hit@4"] = pd.Series(known & (gid[qi] == gid[ci])).groupby(qi).any()[pd.Series(known).groupby(qi).first()].mean()
    for name, v in AGREE.items():
        both = pd.notna(v[qi]) & pd.notna(v[ci])
        m[name] = (v[qi][both] == v[ci][both]).mean()
    m["visual@4"] = np.mean([c in visual_pool[a] for a, c in zip(qi, ci) if a in visual_pool and c in visual_pool])
    per_q = pd.DataFrame({"own": sales[qi], "nb": sales[ci]}).groupby(qi).agg(own=("own", "first"), nb=("nb", "mean")).dropna()
    m["sales_corr"] = per_q["own"].rank().corr(per_q["nb"].rank())
    m["price_ratio"] = np.nanmedian(np.maximum(price[qi], price[ci]) / np.minimum(price[qi], price[ci]))
    m["catalogue"] = np.mean([m[s] for s in ["hit@4", "same_subfamily", "same_colour", "same_finishing", "same_material"]])
    m["visual"], m["commercial"] = m["visual@4"], m["sales_corr"]
    m["objective"] = np.mean([m["catalogue"], m["visual"], m["commercial"]])
    return m

def random_top4(queries, seed=0):
    rng, rows = np.random.default_rng(seed), []
    for pool in CATEGORY_POOLS:
        for qi in queries[np.isin(queries, pool)]:
            allowed = pool[~not_allowed(np.array([qi]), pool)[0]]
            rows += [(qi, r + 1, ci) for r, ci in enumerate(rng.choice(allowed, size=4, replace=False))]
    return pd.DataFrame(rows, columns=["q", "rank", "c"])

SHOW = ["hit@4", "same_family", "same_subfamily", "same_colour", "same_finishing", "same_material", "visual@4",
        "sales_corr", "price_ratio", "catalogue", "objective"]
PCT = {c: "{:.0%}" for c in SHOW if c not in ("sales_corr", "price_ratio", "objective")} | {
    "sales_corr": "{:.3f}", "price_ratio": "{:.2f}", "objective": "{:.3f}"}
results = pd.DataFrame({"random (same category)": evaluate(random_top4(TEST_Q)),
                        "weights set by judgement": evaluate(top_k(W_JUDGEMENT, TEST_Q, k=4))}).T
print(f"tuning half: {len(TUNE_Q):,} queries | test half: {len(TEST_Q):,} queries, "
      f"{(gid[TEST_Q] >= 0).sum():,} of them with a known similar product")
results.loc["baseline (Sprint 2)"] = evaluate(base_nb[base_nb["q"].isin(TEST_Q)])
results[SHOW].round(3)

tuning half: 5,263 queries | test half: 5,292 queries, 2,317 of them with a known similar product


,hit@4,same_family,same_subfamily,same_colour,same_finishing,same_material,visual@4,sales_corr,price_ratio,catalogue,objective
random (same category),0.008,0.256,0.077,0.085,0.374,0.170,0.012,0.001,1.333,0.143,0.052
weights set by judgement,0.868,0.956,0.854,0.568,0.800,0.753,0.366,0.623,1.000,0.768,0.586
baseline (Sprint 2),0.915,0.981,0.902,0.590,0.838,0.739,0.414,0.617,1.000,0.797,0.609


In [15]:
fb_pool = json.loads((ROOT / "outputs" / "phase4" / "feedback_pool.json").read_text(encoding="utf-8"))
latest = {}
for path in sorted((ROOT / "outputs" / "phase4").glob("feedback_*.jsonl")):
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            a = json.loads(line)
            latest[(a["member"], a["query"])] = a
answers = [latest[k] for k in sorted(latest)]

pair_row, sims = {}, []
for pq in fb_pool["queries"]:
    for cand in pq["candidates"]:
        pair_row[(pq["query"], cand["key"])] = len(sims)
        sims.append([np.nan if cand["sim"][b] is None else cand["sim"][b] for b in BLOCKS])
S = np.asarray(sims, float)
H = np.isfinite(S).astype(float)          # 1 where both products have the block
S = np.nan_to_num(S)
held_out = {pq["query"] for pq in fb_pool["queries"] if pq["holdout"]}

def preferences(ans):
    """Every "similar" product beats the similar ones below it and every "not similar" one: (winner, loser) rows."""
    win, lose = [], []
    for a in ans:
        sim = [pair_row[(a["query"], c)] for c in a["similar"]]
        nots = [pair_row[(a["query"], c)] for c in a["not"]]
        for i, x in enumerate(sim):
            for y in sim[i + 1:] + nots:
                win.append(x)
                lose.append(y)
    return np.asarray(win, int), np.asarray(lose, int)

train = [a for a in answers if a["query"] not in held_out]
test = [a for a in answers if a["query"] in held_out]
win, lose = preferences(train)
print(f"{len(answers)} sorted queries by members {sorted({a['member'] for a in answers})} | "
      f"learning: {len({a['query'] for a in train})} queries, {len(win):,} preferences | held out: {len(test)} answers")

138 sorted queries by members [1, 2, 3, 4, 5] | learning: 79 queries, 3,330 preferences | held out: 24 answers


In [16]:
def fit_weights(S_, H_, win_, lose_, w0, steps=1500):
    """Bradley-Terry fit of the block weights on (winner, loser) rows, pulled towards w0 (as in Sprint 2)."""
    def lg(p, theta0):
        theta, gamma = p[:-1], p[-1]
        w, beta = np.exp(theta - theta.max()) / np.exp(theta - theta.max()).sum(), np.exp(gamma)
        den = H_ @ w
        sc = np.divide((H_ * S_) @ w, den, out=np.zeros(len(S_)), where=den > 0)
        d = sc[win_] - sc[lose_]
        g_d = -beta / (1 + np.exp(np.clip(beta * d, -50, 50)))
        ds_dw = np.divide(H_ * S_ - sc[:, None] * H_, den[:, None], out=np.zeros_like(S_), where=den[:, None] > 0)
        g_w = (g_d[:, None] * (ds_dw[win_] - ds_dw[lose_])).sum(0)
        return np.append(w * (g_w - w @ g_w) + (theta - theta0) / 0.5 ** 2, (g_d * d).sum() + (gamma - np.log(20.0)))
    theta0 = np.log(np.maximum(w0, 1e-3) / np.maximum(w0, 1e-3).sum())
    p, m1, m2 = np.append(theta0, np.log(20.0)), np.zeros(len(w0) + 1), np.zeros(len(w0) + 1)
    for t in range(1, steps + 1):
        g = lg(p, theta0)
        m1, m2 = 0.9 * m1 + 0.1 * g, 0.999 * m2 + 0.001 * g ** 2
        p -= 0.05 * (m1 / (1 - 0.9 ** t)) / (np.sqrt(m2 / (1 - 0.999 ** t)) + 1e-8)
    e = np.exp(p[:-1] - p[:-1].max())
    return e / e.sum()

def per_query_right(w, S_, ans):
    """Right / total preferences per query for fixed weights."""
    sc = model_scores(w, S_, H)
    out = {}
    for a in ans:
        hw, hl = preferences([a])
        o = out.setdefault(a["query"], [0, 0])
        o[0] += int((sc[hw] > sc[hl]).sum())
        o[1] += len(hw)
    return pd.DataFrame(out, index=["right", "pairs"]).T

def model_scores(w, s, h):
    den = h @ w
    return np.divide((h * s) @ w, den, out=np.zeros(len(s)), where=den > 0)

QUERIES = sorted({a["query"] for a in answers})
FOLD = dict(zip(QUERIES, np.random.default_rng(1).integers(0, 5, len(QUERIES))))
W_START = np.array([fb_pool["weights"][b] for b in BLOCKS])
W_BASE_V = np.array([W_BASE[b] for b in BLOCKS])

def team_score(S_):
    """Team preferences ordered right: fixed baseline weights, and 5-fold CV with refitted weights."""
    fixed = per_query_right(W_BASE_V, S_, answers)
    cv = []
    for f in range(5):
        tr = [a for a in answers if FOLD[a["query"]] != f]
        w_ = fit_weights(S_, H, *preferences(tr), W_START)
        cv.append(per_query_right(w_, S_, [a for a in answers if FOLD[a["query"]] == f]))
    return fixed, pd.concat(cv)

def boot_diff(a, b, n=2000, seed=2):
    """95% interval of (accuracy of b) - (accuracy of a), resampling queries."""
    b = b.loc[a.index]
    rng, d = np.random.default_rng(seed), []
    for _ in range(n):
        i = rng.integers(0, len(a), len(a))
        d.append(b["right"].values[i].sum() / b["pairs"].values[i].sum() - a["right"].values[i].sum() / a["pairs"].values[i].sum())
    return np.percentile(d, [2.5, 97.5])

acc = lambda t: t["right"].sum() / t["pairs"].sum()
base_fixed, base_cv = team_score(S)
print(f"baseline on the team's preferences: fixed weights {acc(base_fixed):.1%}, 5-fold CV {acc(base_cv):.1%} "
      f"({base_fixed['pairs'].sum():,} preferences, {len(base_fixed)} queries)")

baseline on the team's preferences: fixed weights 70.5%, 5-fold CV 72.8% (4,029 preferences, 99 queries)


## 3. Explanations for every similar product

The brief asks, for each product, for at least 4 similar products **and the characteristics that explain the match**,
so a buyer can check it. Because the score is a weighted mean of block similarities, and each block is a dot product of
rows of length 1 split into named slices (section 1), the explanation can be an **exact decomposition** instead of an
approximation:

- score = Σ over blocks of w_b · sim_b / Σ w_b → **the share of each block**;
- sim_categorical = Σ over attributes of the slice products → **which attributes are shared, and how many points each adds**;
- sim_composition = Σ over materials → **shared materials** (rare ones weigh more, IDF);
- sim_numeric = price part + launch part + season part.

Differences are added from the data (price, colour, finish, sub-family). Two different colour names that look alike on the
photos (Sprint 2, section 9) are reported as **close colours** instead of different. No surrogate model, no sampling and no
LLM are involved, so the explanation cannot say something the score does not contain.

In [17]:
ATTR_LABEL = {"GFA": "family", "GFS": "sub-family", "CATEGORY_MATRIX": "planning category", "CLR_DES": "colour",
              "CLR_TYPE": "colour type", "FINISHING": "finish", "FINISH_FAMILY": "metal", "MATERIAL": "main fabric",
              "PRINT_TYPE": "print", "DIMENSION": "size type", "OUTFIT": "outfit", "THEME": "theme",
              "FASHIONTYPE": "fashion type", "PROD_SEG": "segment", "NUMBER_OF_UNITS": "units"}
BLOCK_LABEL = {"image": "photo", "categorical": "catalogue attributes", "text": "description", "composition": "materials",
               "palette": "photo colours", "numeric": "price and launch"}
SLICES = {"categorical": cat_attrs, "composition": comp_attrs, "numeric": num_attrs}
VALUE_COL = {a: cols[-1] for a, (cols, _) in CATEGORICAL.items()}

# colour names measured on the photos (Sprint 2, section 9): reference LAB per name, similarity of two names
ref = pd.read_csv(ROOT / "outputs" / "phase2a" / "colour_reference.csv").dropna(subset=["L"])
lab_ref = ref[["L", "a", "b"]].to_numpy(float) * np.array([0.5, 1, 1])
csim = np.exp(-((lab_ref[:, None] - lab_ref[None]) ** 2).sum(-1) / (2 * 8.0 ** 2))
COLOUR_SIM = {(x, y): csim[i, j] for i, x in enumerate(ref["CLR_DES"]) for j, y in enumerate(ref["CLR_DES"]) if i != j}

def parts(block, i, j):
    m = blocks[block]
    return {a: float(m[i, s:e] @ m[j, s:e]) for a, (s, e) in SLICES[block].items() if abs(m[i, s:e] @ m[j, s:e]) > 1e-9}

def explain(w, i, j):
    """Exact explanation of the score of the pair (query i, candidate j)."""
    contrib = contributions(w, i, j)
    score = sum(contrib.values())
    out = {"score": score, "shares": {BLOCK_LABEL[b]: v / score for b, v in contrib.items()}}
    shared = []
    if "categorical" in contrib:
        sim = float(blocks["categorical"][i] @ blocks["categorical"][j])
        for a, v in sorted(parts("categorical", i, j).items(), key=lambda kv: -kv[1]):
            if a != "CAT":
                shared.append((ATTR_LABEL.get(a, a), items[VALUE_COL[a]].iat[i], contrib["categorical"] * v / sim))
    out["shared"] = shared
    if "composition" in contrib:
        sim = float(comp[i] @ comp[j])
        out["materials"] = [(m_, contrib["composition"] * v / sim) for m_, v in sorted(parts("composition", i, j).items(), key=lambda kv: -kv[1])]
    a_, b_ = items.iloc[i], items.iloc[j]
    diffs = []
    if abs(b_["PRICE_BASE_W_VAT"] - a_["PRICE_BASE_W_VAT"]) > 0.05 * a_["PRICE_BASE_W_VAT"]:
        diffs.append(f"price {b_['PRICE_BASE_W_VAT']:.2f} € vs {a_['PRICE_BASE_W_VAT']:.2f} €")
    if a_["CLR_DES"] != b_["CLR_DES"]:
        s_ = COLOUR_SIM.get((a_["CLR_DES"], b_["CLR_DES"]))
        diffs.append(f"close colours {b_['CLR_DES']} ≈ {a_['CLR_DES']} ({s_:.2f})" if s_ is not None and s_ >= 0.5
                     else f"colour {b_['CLR_DES']} vs {a_['CLR_DES']}")
    for col, lab in [("GFS_DES_EN", "sub-family"), ("FINISHING", "finish")]:
        if pd.notna(a_[col]) and pd.notna(b_[col]) and a_[col] != b_[col]:
            diffs.append(f"{lab} {b_[col]} vs {a_[col]}")
    out["differences"] = diffs
    return out

def summary(e):
    lead = ", ".join(f"{k} {v:.0%}" for k, v in sorted(e["shares"].items(), key=lambda kv: -kv[1])[:3])
    facts = [f"same {a}: {v}" for a, v, _ in e["shared"][:3]]
    if e.get("materials"):
        facts.append("shared materials: " + ", ".join(m_ for m_, _ in e["materials"][:3]))
    return f"{lead}. " + "; ".join(facts) + (". Differs: " + "; ".join(e["differences"][:2]) if e["differences"] else "")

for t in base_nb[(base_nb["q"] == q) & (base_nb["rank"] <= 5)].itertuples():
    print(f"{t.rank}. {items[KEY].iat[t.c]}  {t.score:.3f}  {summary(explain(W_BASE, q, t.c))}")

1. 210347_NV  0.880  photo 44%, catalogue attributes 34%, description 9%. same family: Earrings; same sub-family: Dangle; same planning category: 52 - Earrings Long; shared materials: Glass, Iron, Polyester. Differs: colour Navy vs Burgundy
2. 210347_OR  0.832  photo 46%, catalogue attributes 32%, description 9%. same family: Earrings; same sub-family: Dangle; same planning category: 52 - Earrings Long; shared materials: Glass, Iron, Polyester. Differs: colour Orange vs Burgundy
3. 214234_BU  0.819  photo 46%, catalogue attributes 35%, description 9%. same family: Earrings; same sub-family: Dangle; same colour: Burgundy. Differs: price 7.99 € vs 8.99 €; finish Black Plating vs Golden
4. 210305_LL  0.809  photo 45%, catalogue attributes 33%, description 9%. same family: Earrings; same sub-family: Dangle; same planning category: 52 - Earrings Long; shared materials: Glass, Iron, Polyester. Differs: colour Lilac vs Burgundy
5. 210347_TG  0.804  photo 46%, catalogue attributes 33%, descrip

The brief's example, explained. The photo and the catalogue attributes carry most of each score; the matches share the
family, sub-family and planning category with the query and its materials (glass, iron, polyester), and differ in colour.

**Check 1: completeness.** The parts must add up to the score, for every pair.

In [18]:
rng = np.random.default_rng(0)
sample = base_nb[base_nb["rank"] <= 5].sample(2000, random_state=0)
err_blocks = err_attrs = 0.0
for t in sample.itertuples():
    e = explain(W_BASE, t.q, t.c)
    err_blocks = max(err_blocks, abs(e["score"] - t.score))
    sim = float(blocks["categorical"][t.q] @ blocks["categorical"][t.c])
    err_attrs = max(err_attrs, abs(sum(parts("categorical", t.q, t.c).values()) - sim))
print(f"largest gap on 2,000 pairs: blocks vs score {err_blocks:.1e} | attributes vs categorical similarity {err_attrs:.1e}")

largest gap on 2,000 pairs: blocks vs score 8.5e-07 | attributes vs categorical similarity 1.0e-07


**Check 2: faithfulness (deletion test).** If the explanation names the block that really drives a match, removing that
block (weight 0) should push the match down the ranking more than removing the block it names last, or a random block.
Measured on the best match of 300 random products.

In [19]:
rows = []
for qi in rng.choice(len(items), 300, replace=False):
    first = base_nb[(base_nb["q"] == qi) & (base_nb["rank"] == 1)]
    if first.empty:
        continue
    ci = int(first["c"].iat[0])
    ranked = [b for b, _ in sorted(contributions(W_BASE, qi, ci).items(), key=lambda kv: -kv[1])]
    if len(ranked) < 2:
        continue
    pool = next(p for p in CATEGORY_POOLS if qi in p)
    for label, b in [("named first", ranked[0]), ("named last", ranked[-1]), ("random", ranked[rng.integers(len(ranked))])]:
        s = score_matrix({**W_BASE, b: 0.0}, np.array([qi]), pool)[0].astype(float)
        s[not_allowed(np.array([qi]), pool)[0]] = -np.inf
        rows.append({"block removed": label, "new rank": int((s > s[pool == ci][0]).sum()) + 1})
dt = pd.DataFrame(rows).groupby("block removed")["new rank"]
pd.DataFrame({"mean new rank": dt.mean(), "share no longer first": dt.apply(lambda r: (r > 1).mean())}).round(2)

,mean new rank,share no longer first
block removed,,
named first,6.49,0.43
named last,1.30,0.17
random,2.23,0.28


**Check 3: agreement with the team.** For the candidates the team sorted, the mean similarity of each block for the ones
called *similar* and *not similar*, and the difference in standard deviations. Blocks that separate the two groups most
are the ones worth naming first.

In [20]:
rows = []
for a in answers:
    for label, keys_ in (("similar", a["similar"]), ("not similar", a["not"])):
        for c in keys_:
            r_ = pair_row[(a["query"], c)]   # a block missing for one of the two products is left out, not counted as 0
            rows.append({"verdict": label, **{BLOCK_LABEL[b]: (S[r_, k] if H[r_, k] else np.nan) for k, b in enumerate(BLOCKS)}})
ag = pd.DataFrame(rows)
tab = ag.groupby("verdict").mean().T
tab["difference / sd"] = (tab["similar"] - tab["not similar"]) / ag.drop(columns="verdict").std()
tab.sort_values("difference / sd", ascending=False).round(3)

verdict,not similar,similar,difference / sd
catalogue attributes,0.411,0.744,1.344
description,0.672,0.854,1.089
photo,0.804,0.885,1.073
materials,0.503,0.778,0.789
price and launch,0.440,0.668,0.751
photo colours,0.372,0.512,0.352


The catalogue attributes, the description and the photo separate what the team calls similar best; the colours of the
photo least (consistent with the brief's own example, where the four similar earrings all have other colours).

## 4. Price and sales estimates for a new product

The brief's business problem (slide 4) is buying the right quantity of a **new** product and sending it to the right
stores. The engine finds similar products; here they and all the product's attributes become an estimate of its sales and
of its price, each with an explanation.

**First, the data the estimates rest on** (the questions raised for Sprint 3): are there products without a price, at 0 €
or with a missing value? And what is missing in the sales?

In [21]:
price_ = items["PRICE_BASE_W_VAT"].astype(float)
qty = items["SALES_QTY"]
print(f"list price: {price_.isna().sum()} missing, {(price_ <= 0).sum()} at 0 € or less, "
      f"{price_.nunique()} price points from {price_.min():.2f} € to {price_.max():.2f} €")
print(f"sales: {qty.isna().sum()} colourways without a sales row, {(qty == 0).sum()} with 0 units "
      f"({((qty == 0) & (items['SALES_AMT_FX_RATE'] > 0)).sum()} of them with a positive amount)")
print(f"landed cost (PRICE_LCP): missing for {pd.to_numeric(items['PRICE_LCP'], errors='coerce').isna().sum()} colourways")
items["launch"] = pd.to_datetime(items["STORE_DATE_FINAL"])
items["days_on_sale"] = (items["launch"].max() - items["launch"]).dt.days.astype(float)
items["log_sales"] = np.log1p(qty.where(qty > 0))
print("median units by season:", items.groupby("SEA_COD")["SALES_QTY"].median().to_dict())
v = items.dropna(subset=["log_sales"])
within = [g["log_sales"].rank().corr(g["days_on_sale"].rank()) for _, g in v.groupby(["SEA_COD", "CAT_DES_EN"]) if len(g) > 30]
print(f"inside a season and category, units vs days on sale: median Spearman {np.median(within):+.2f}")

list price: 0 missing, 0 at 0 € or less, 40 price points from 3.99 € to 250.00 €
sales: 370 colourways without a sales row, 9 with 0 units (1 of them with a positive amount)
landed cost (PRICE_LCP): missing for 168 colourways
median units by season: {'232': 3377.0, '241': 3637.0, '242': 1336.0}
inside a season and category, units vs days on sale: median Spearman +0.26


- **Every product has a list price** (no 0 € and no missing value), so the price estimate can use all of them. What is
  missing is the landed cost, and the sales of 370 products (no sales row) plus 9 with 0 units. A product without sales is
  *unknown*, not zero: it stays out of training and evaluation (and still gets an estimate: these are exactly the new
  products a buyer must size).
- **Sales are accumulated until the extraction**, so the newest season (242) had much less time to sell, and later
  launches sold less even inside a season. Hence a **temporal test**: train on seasons 232 and 241, estimate 242, only on
  models never sold before. And the honest sales metric compares only products **launched in the same month**, so the time
  on sale cannot help.
- **Buyer decisions are left out**: `PUR_BET` (purchase depth) and `TYPOLOGY` (store tiers) are the buyer's own expectation
  of the sales; using them to estimate the sales would be circular. The landed cost is left out of the price (price = cost ×
  markup).

In [22]:
import lightgbm as lgb

FC_CATEGORICAL = ["CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN", "CATEGORY_MATRIX", "CLR_DES", "CLR_TYPE", "FINISHING",
                  "MATERIAL", "PRINT_TYPE", "DIMENSION", "OUTFIT", "FASHIONTYPE", "PROD_SEG", "NUMBER_OF_UNITS",
                  "COLLECTION_TYPE", "MADE_IN", "SUP_COD"]
LGB = dict(objective="regression_l2", learning_rate=0.03, num_leaves=31, min_child_samples=20, feature_fraction=0.8,
           bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, n_estimators=600, random_state=0, deterministic=True,
           force_col_wise=True, num_threads=4, verbose=-1)
old_season = items["SEA_COD"].astype(str).ne("242").to_numpy()
seen = set(items.loc[old_season, "PROD_REF"].astype(str))
TRAIN, TEST = old_season, ~old_season & ~items["PROD_REF"].astype(str).isin(seen).to_numpy()

def pcs(m, k=16):
    """Principal components of an embedding, fitted on the training rows only; no embedding -> missing."""
    has_ = np.abs(m).sum(1) > 0
    t = m[TRAIN & has_]
    mu = t.mean(0)
    _, _, vt = np.linalg.svd(t - mu, full_matrices=False)
    out = (m - mu) @ vt[:k].T
    out[~has_] = np.nan
    return out

def features(target):
    X = pd.DataFrame({c: pd.Categorical(items[c].astype("string")) for c in FC_CATEGORICAL})
    X["n_sizes"] = items["n_skus"].astype(float)
    X["net_weight"] = pd.to_numeric(items["NET_WEIGHT"], errors="coerce")
    X["launch_month"] = items["launch"].dt.month.astype(float)
    X["days_on_sale"] = items["days_on_sale"]
    if target == "sales":
        X["price"] = price_
        X["price_pct_in_cat"] = items.groupby("CAT_DES_EN")["PRICE_BASE_W_VAT"].rank(pct=True)
    for name, m in [("img", image_m), ("txt", text_m)]:
        for j, col in enumerate(pcs(m).T):
            X[f"{name}_pc{j + 1}"] = col
    for j in np.argsort(-(comp[TRAIN] > 0).sum(0))[:20]:
        X[f"mat_{j}"] = (comp[:, j] > 0).astype(float)
    return X

FAMILIES = [("img_pc", "photo"), ("txt_pc", "description"), ("mat_", "materials"), ("price", "price"),
            ("days_on_sale", "time on sale"), ("launch_month", "launch month"), ("n_sizes", "number of sizes"),
            ("net_weight", "weight")]
family = lambda c: next((f for p, f in FAMILIES if c.startswith(p)), "attribute: " + c)

def spearman(a, b):
    if np.ptp(a) == 0 or np.ptp(b) == 0:
        return 0.0
    r = pd.Series(a).rank().corr(pd.Series(b).rank())
    return 0.0 if pd.isna(r) else float(r)

### 4.1 Sales

Estimates compared, all on the same test products: the median of the category; the median of the same family and price
band; the **similar past products** found by the engine (similarity-weighted mean of the 10 most similar products of the
older seasons, other models only); and a **gradient-boosting model** (LightGBM) on the product's attributes, price, time on
sale, materials and the principal components of its photo and description embeddings.

In [23]:
y = items["log_sales"].to_numpy(float)
fit = TRAIN & np.isfinite(y)
ev_rows = TEST & np.isfinite(y)
month_group = (items["CAT_DES_EN"].astype(str) + "|" + items["launch"].dt.month.astype(str)).to_numpy()
cats = items["CAT_DES_EN"].to_numpy()

def sales_metrics(p, rows=ev_rows):
    yy, pp, cc, gg = y[rows], p[rows], cats[rows], month_group[rows]
    within = np.mean([spearman(yy[cc == c], pp[cc == c]) for c in np.unique(cc)])
    month = [(spearman(yy[gg == g], pp[gg == g]), (gg == g).sum()) for g in np.unique(gg) if (gg == g).sum() > 10]
    top_true = pd.Series(yy).groupby(cc).rank(pct=True).to_numpy() > 0.8
    top_pred = pd.Series(pp + np.random.default_rng(0).random(len(pp)) * 1e-9).groupby(cc).rank(pct=True).to_numpy() > 0.8
    return {"ranking within category": within,
            "ranking, same launch month": np.average([m for m, _ in month], weights=[n for _, n in month]),
            "top-20% precision": top_true[top_pred].mean()}

def median_by(cols):
    d = items[cols].astype("string").assign(y=np.where(fit, y, np.nan))
    return d.groupby(cols, dropna=False)["y"].transform("median").fillna(d.groupby("CAT_DES_EN")["y"].transform("median")).to_numpy(float)

def analogue(target, w, k=10):
    """Similarity-weighted mean of the target over the k most similar products of the training seasons (other models)."""
    est = np.full(len(items), np.nan)
    ref_ = items["PROD_REF"].astype(str).to_numpy()
    for pool in CATEGORY_POOLS:
        cand = pool[fit[pool]] if target is y else pool[TRAIN[pool]]
        for start in range(0, len(pool), 1024):
            qs = pool[start:start + 1024]
            s = score_matrix(w, qs, cand).astype(float)
            s[ref_[qs][:, None] == ref_[cand][None, :]] = -np.inf
            top = np.argsort(-s, axis=1, kind="stable")[:, :k]
            ww = np.clip(np.take_along_axis(s, top, axis=1), 0, None)
            est[qs] = (ww * target[cand][top]).sum(1) / np.maximum(ww.sum(1), 1e-12)
    return est

band = pd.qcut(items["PRICE_BASE_W_VAT"].rank(method="first"), 5, labels=False).astype(str)
items["price_band"] = band
X_sales = features("sales")
sales_model = lgb.LGBMRegressor(**LGB).fit(X_sales[fit], y[fit])
est = {"category median": median_by(["CAT_DES_EN"]),
       "family + price band median": median_by(["CAT_DES_EN", "GFA_DES_EN", "price_band"]),
       "similar past products (engine)": analogue(y, W_BASE),
       "gradient boosting": sales_model.predict(X_sales)}
print(f"training {fit.sum():,} products (seasons 232, 241) | test {ev_rows.sum():,} new products (season 242)")
pd.DataFrame({n: sales_metrics(p) for n, p in est.items()}).T.round(3)

training 6,567 products (seasons 232, 241) | test 3,393 new products (season 242)


,ranking within category,"ranking, same launch month",top-20% precision
category median,0.000,0.000,0.209
family + price band median,0.263,0.198,0.335
similar past products (engine),0.332,0.283,0.365
gradient boosting,0.574,0.408,0.381


The gradient-boosting model ranks the new products of a category clearly better than the baselines, also among products
launched in the same month (where the time on sale cannot help): about 0.41 against 0.28 for the similar products and 0.20
for family + price band. The 95% intervals (bootstrap over models, `outputs/phase5b/forecast_report.md`) do not include 0.
Adding the similar products as a feature of the model did not help significantly (−0.004 to +0.018), so the simpler model
is kept, and the similar products are shown to the buyer as a reference instead.

**Explanation of a sales estimate: SHAP values.** For tree models, TreeSHAP computes the exact Shapley values (how much each
feature moved this estimate away from the average) in milliseconds, so every product gets them. They are grouped into
business families, because a single principal component of the photo means nothing to a buyer.

In [24]:
def shap_families(model, X, rows):
    sv = pd.DataFrame(model.predict(X[rows], pred_contrib=True), columns=list(X.columns) + ["expected"]).drop(columns="expected")
    return sv.T.groupby(lambda c: family(c)).sum().T

fam_sales = shap_families(sales_model, X_sales, ev_rows)
print("what drives the sales estimates (mean |SHAP|, log units, test products):")
print(fam_sales.abs().mean().sort_values(ascending=False).head(10).round(3).to_string())
k_ex = items[KEY].to_numpy()[ev_rows][7]
i_ex = int(np.flatnonzero(items[KEY].to_numpy() == k_ex)[0])
contrib_ex = shap_families(sales_model, X_sales, np.arange(len(items)) == i_ex).iloc[0]
print(f"\n{k_ex} ({items['GFS_DES_EN'].iat[i_ex]}, {items['CLR_DES'].iat[i_ex]}): sold {int(qty.iat[i_ex]):,}, "
      f"estimate {np.expm1(est['gradient boosting'][i_ex]):,.0f} units")
print(contrib_ex.reindex(contrib_ex.abs().sort_values(ascending=False).index).head(5).round(2).to_string())

what drives the sales estimates (mean |SHAP|, log units, test products):
attribute: FASHIONTYPE        0.331
photo                         0.327
attribute: COLLECTION_TYPE    0.222
attribute: CATEGORY_MATRIX    0.134
attribute: SUP_COD            0.118
attribute: CLR_DES            0.094
time on sale                  0.064
description                   0.063
weight                        0.058
attribute: GFS_DES_EN         0.057

153795_NM (Pin, Neutral Multicolor): sold 2,366, estimate 2,291 units
photo                        -0.22
attribute: FASHIONTYPE        0.22
attribute: COLLECTION_TYPE    0.12
attribute: CATEGORY_MATRIX    0.08
time on sale                 -0.08


### 4.2 Price

Same test, target log(price). The model never sees the price, the price part of the engine (the numeric block) or the cost.
A useful business check: the estimate rounded to the nearest of the 40 list prices is the right one or its neighbour.

In [25]:
yp = np.log(price_.to_numpy(float))
points = np.sort(price_.unique())
nearest = lambda v: np.abs(v[:, None] - points[None, :]).argmin(1)

def price_metrics(p, rows=TEST):
    ye, pe = np.exp(yp[rows]), np.exp(p[rows])
    steps = np.abs(nearest(pe) - nearest(ye))
    return {"mean error €": np.mean(np.abs(ye - pe)), "exact list price": (steps == 0).mean(), "right or next": (steps <= 1).mean()}

def median_price(cols):
    d = items[cols].astype("string").assign(y=np.where(TRAIN, yp, np.nan))
    return d.groupby(cols, dropna=False)["y"].transform("median").fillna(d.groupby("CAT_DES_EN")["y"].transform("median")).to_numpy(float)

X_price = features("price")
price_model = lgb.LGBMRegressor(**LGB).fit(X_price[TRAIN], yp[TRAIN])
est_p = {"category median": median_price(["CAT_DES_EN"]),
         "sub-family median": median_price(["CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN"]),
         "similar past products (engine, no price block)": analogue(yp, {**W_BASE, "numeric": 0.0}),
         "gradient boosting": price_model.predict(X_price)}
print(pd.DataFrame({n: price_metrics(p) for n, p in est_p.items()}).T.round(3).to_string())
print("\nwhat drives the price estimates (mean |SHAP|, log units):")
print(shap_families(price_model, X_price, TEST).abs().mean().sort_values(ascending=False).head(6).round(3).to_string())

                                                mean error €  exact list price  right or next
category median                                        5.761             0.232          0.446
sub-family median                                      4.160             0.332          0.597
similar past products (engine, no price block)         3.852             0.300          0.668
gradient boosting                                      3.471             0.313          0.714

what drives the price estimates (mean |SHAP|, log units):


attribute: CATEGORY_MATRIX    0.296
weight                        0.155
attribute: CAT_DES_EN         0.124
attribute: GFS_DES_EN         0.049
attribute: GFA_DES_EN         0.047
attribute: FASHIONTYPE        0.036


The model gets the right list price or its neighbour for about 71% of the new products (sub-family median: 60%), with a
mean error of about 3.5 €. The planning category and the weight drive the estimate most, then the category and the photo.

## 5. Improvements tried against the baseline

Each idea came from a measured weakness. It is kept only if it wins on signals it was not tuned on and no guard drops by
more than 1 point.

### 5.1 Fusion: standardise each block's similarity per category — rejected

The image similarities are compressed (a random pair of the same category already scores about 0.8, the best matches about
0.93), while the categorical similarity spans 0–1, so the weights do not mean the same for every block. Idea: subtract the
mean and divide by the standard deviation of random pairs of the same category, per block, before the weighted sum.

In [26]:
rng = np.random.default_rng(0)
stats = {}
for pool in CATEGORY_POOLS:
    cat_name = items["CAT_DES_EN"].iat[pool[0]]
    a, b = rng.choice(pool, 20000), rng.choice(pool, 20000)
    keep = items["PROD_REF"].to_numpy()[a] != items["PROD_REF"].to_numpy()[b]
    a, b = a[keep], b[keep]
    for bl in BLOCKS:
        both = (has[bl][a] > 0) & (has[bl][b] > 0)
        v = np.einsum("ij,ij->i", blocks[bl][a[both]], blocks[bl][b[both]])
        stats[(cat_name, bl)] = (v.mean(), v.std())
print(pd.DataFrame({bl: [round(stats[(c, bl)][0], 2) for c in sorted(items["CAT_DES_EN"].unique())] for bl in BLOCKS},
                   index=sorted(items["CAT_DES_EN"].unique())).to_string(), "\n(mean similarity of random pairs)")
pair_q = np.zeros(len(S), int)
pair_c = np.zeros(len(S), int)
pos_ = pd.Series(np.arange(len(items)), index=items[KEY])
for (qk, ck), r_ in pair_row.items():
    pair_q[r_], pair_c[r_] = pos_[qk], pos_[ck]
qcat = items["CAT_DES_EN"].to_numpy()[pair_q]
Z = S.copy()
for k_, bl in enumerate(BLOCKS):
    mu = np.array([stats[(c, bl)][0] for c in qcat])
    sd = np.array([stats[(c, bl)][1] for c in qcat])
    Z[:, k_] = np.where(H[:, k_] > 0, (S[:, k_] - mu) / sd, 0)
z_fixed, z_cv = team_score(Z)
pd.DataFrame({"fixed weights": [acc(base_fixed), acc(z_fixed)], "5-fold CV refit": [acc(base_cv), acc(z_cv)],
              "CV gain, 95% CI": ["", "{:+.3f} to {:+.3f}".format(*boot_diff(base_cv, z_cv))]},
             index=["raw cosine (baseline)", "standardised per category"]).round(3)

           image  categorical  text  composition  palette  numeric
Apparel     0.81         0.26  0.64         0.21     0.16     0.31
Footwear    0.79         0.30  0.62         0.67     0.31     0.30
Hand Bag    0.83         0.26  0.62         0.56     0.22     0.31
Jewellery   0.75         0.33  0.56         0.20     0.36     0.31
Wallet      0.80         0.25  0.61         0.62     0.17     0.31 
(mean similarity of random pairs)


,fixed weights,5-fold CV refit,"CV gain, 95% CI"
raw cosine (baseline),0.705,0.728,
standardised per category,0.716,0.703,-0.049 to -0.001


Worse with cross-validation (the interval excludes 0). The team-fitted weights already compensate the compressed photo scale
(the team gave the photo 0.62 of the weight). The literature on hybrid retrieval agrees: a weighted combination tuned on
validation data beats rank fusion, which is the choice when there is nothing to tune with. **Not adopted.**

### 5.2 Less weight on colour and theme — rejected

The brief's four similar products for `167718_BU` have other colours and another theme, and the baseline ranks them far
down. Hypothesis: colour (weight 2) and theme (1.5) weigh too much inside the categorical block.

In [27]:
cat_base = cat_m.copy()
BRIEF = ["169597_FU", "169607_IG", "169607_PE", "177781_LB"]
brief_idx = [int(pos_[k]) for k in BRIEF]

def brief_ranks(w):
    s = score_matrix(w, np.array([q]), next(p for p in CATEGORY_POOLS if q in p))[0].astype(float)
    pool = next(p for p in CATEGORY_POOLS if q in p)
    s[not_allowed(np.array([q]), pool)[0]] = -np.inf
    return [int((s > s[pool == j][0]).sum()) + 1 for j in brief_idx]

rows = {}
for name, factors in [("baseline", {}), ("colour ×0.5", {"CLR_DES": 0.5}), ("colour 0", {"CLR_DES": 0}),
                      ("theme 0", {"THEME": 0}), ("colour 0 + theme 0", {"CLR_DES": 0, "THEME": 0})]:
    m = cat_base.copy()
    for a, f in factors.items():
        s0_, e0_ = cat_attrs[a]
        m[:, s0_:e0_] *= np.sqrt(f)
    blocks["categorical"] = normalise_rows(m)
    S2 = S.copy()
    S2[:, BLOCKS.index("categorical")] = np.einsum("ij,ij->i", blocks["categorical"][pair_q], blocks["categorical"][pair_c])
    fx, cv = team_score(S2)
    g = evaluate(top_k(W_BASE, TEST_Q, k=4))
    rows[name] = {"team, fixed": acc(fx), "team, CV": acc(cv), "brief ranks": brief_ranks(W_BASE),
                  "same sub-family": g["same_subfamily"], "same material": g["same_material"], "sales corr.": g["sales_corr"]}
blocks["categorical"] = cat_base
pd.DataFrame(rows).T

,"team, fixed","team, CV",brief ranks,same sub-family,same material,sales corr.
baseline,0.705138,0.728469,"[452, 44, 66, 658]",0.901927,0.739319,0.617355
colour ×0.5,0.697195,0.717051,"[472, 53, 79, 683]",0.919596,0.745308,0.616036
colour 0,0.687267,0.708116,"[494, 60, 91, 699]",0.932162,0.76,0.613876
theme 0,0.701167,0.736907,"[484, 56, 83, 690]",0.922997,0.763585,0.57811
colour 0 + theme 0,0.674113,0.706875,"[525, 80, 113, 729]",0.949027,0.816575,0.569606


The team's agreement falls when colour loses weight, the brief's products move further down, and theme 0 costs about
4 points of sales correlation for a team gain inside the noise (±2.5 points). The team values colour; the brief's example
differs for other reasons. **Not adopted.**

### 5.3 Image encoder — FashionSigLIP adopted, DINOv2 rejected

The photo is the block the team weighs most, so a better image representation is where a gain can come from. Three
encoders were run once on the same item crops (CPU, 40–60 min each, outside this notebook) and compared with the Sprint 2
CLIP ViT-B/32 on the team's preferences, on labels the encoders never saw (sub-family, material, finish and colour of a
photo's 4 nearest photos, test half), on the brief's example and on the guards. The saved comparison is loaded:

In [28]:
enc = pd.read_csv(ROOT / "outputs" / "phase2b" / "image_encoders.csv").set_index("encoder")
enc[["team pairs (current w)", "team pairs (CV refit)", "img_same_subfamily", "img_same_material", "img_same_finish",
     "img_same_colour", "brief ranks (image alone)", "hit@4", "sales_corr"]].round(3)

,team pairs (current w),team pairs (CV refit),img_same_subfamily,img_same_material,img_same_finish,img_same_colour,brief ranks (image alone),hit@4,sales_corr
encoder,,,,,,,,,
CLIP B/32 (baseline),0.705,0.728,0.577,0.303,0.585,0.433,"[5, 10, 39, 214]",0.915,0.618
FashionCLIP,0.720,0.736,0.650,0.357,0.602,0.429,"[3, 7, 49, 225]",0.898,0.611
FashionSigLIP,0.721,0.740,0.681,0.391,0.619,0.434,"[2, 12, 33, 68]",0.910,0.614
DINOv2-small,0.708,0.699,0.639,0.360,0.544,0.254,"[6, 50, 19, 53]",0.801,0.571


- **DINOv2** (self-supervised, shape and texture) loses colour (25% of the nearest photos share the colour, against 43%) and
  the team's agreement: rejected. This matches published results on fashion product matching, where CLIP-type encoders beat
  DINOv2.
- **FashionSigLIP** (trained on fashion product data) is best on every label the encoders never saw: +10 points of
  sub-family and +9 of material from the photo alone, over more than 5,000 queries, and it puts the brief's four products at
  ranks 2, 12, 33 and 68 by the photo alone. On the team's preferences it gains 1.2–1.5 points: positive in 88–95% of the
  bootstrap resamples, but not significant at 95% with 138 sorted queries. The guards stay within 1 point. **Adopted** for
  the improved version (`data/embeddings/image_fashionsiglip.npy`), the CLIP baseline being kept for comparison.

## 6. Photos of a person wearing the item — *in progress*

Sprint 1 cropped a photo on a white background to its non-white pixels. When a model wears the item, that crop is the whole
person, so the image vector describes the outfit: the swimsuit `211451_BM` is embedded as white trousers. These photos are
being found (CLIP on the whole photo as a cheap filter, then the OWLv2 detector for a person and for the expected item),
re-cropped to the item and embedded again. This section will show the calibration by eye, the effect on the affected
products and the decision.

## Summary

| | Result |
|---|---|
| Baseline | Sprint 2 engine rebuilt: same numbers as the pipeline (`hit@4` 91.5%, team preferences 72.8% in cross-validation) |
| Explanations | Exact decomposition of every match (block shares, shared attributes and materials, differences, close colours); complete to 1e-6; faithful (removing the block named first moves the match most); the blocks named first are the ones that separate the team's judgements |
| Price | Every product has a price. Estimate for new products: right list price or its neighbour 71% (sub-family median 60%), mean error ≈ 3.5 € |
| Sales | Ranking of new products among those launched in the same month: 0.41 (similar products 0.28, family + price 0.20); explanations by exact SHAP values |
| Kept | FashionSigLIP image encoder |
| Rejected, with evidence | Standardised fusion, lower colour/theme weights, DINOv2 |